# Component 1 (C1): DistilRoBERTa Training on Google Colab (GPU)

**Project:** J26-IT-349: AI-Driven Early Warning System for Student Dropout in Higher Education (SLIIT)  
**Component:** C1 - Student Affective State Detection  
**Model:** `distilroberta-base` (Multi-Label 5 Academic Distress Classes with BCEWithLogitsLoss + PosWeight)  
**Owner:** Nidu  

---

## 1. Instructions: What to Upload to Google Drive

1. In your Google Drive, create a folder named `J26-IT-349/Nidu-c1-affective/`.
2. Upload the following files/folders from your local `components/Nidu-c1-affective/`:
   - `data/processed/` (contains `train.csv`, `val.csv`, `test_proxy.csv`)
   - `src/` (contains `train.py`, `evaluation.py`, `baselines.py`, `config.yaml`, `label_map.yaml`, `data/`)
   - `requirements.txt`

## 2. Instructions: What to Download Back to Local Machine

After training finishes, download these files back to `components/Nidu-c1-affective/`:
- `models/checkpoint-best/` (the trained model weights and tokenizer)
- `models/thresholds.json` (tuned decision thresholds per class)
- `reports/metrics/distilroberta_full.json` (and `ablation_distilroberta_goemotions.json`)
- `reports/figures/confusion_matrix_distilroberta.png`

> **CRITICAL:** Do NOT install PyTorch from `requirements-lock.txt`! Colab comes with a pre-installed CUDA GPU PyTorch build. Overwriting it with a CPU lock file will disable GPU acceleration.

In [ ]:
# Check GPU Availability in Colab
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.")

## 2. Mount Google Drive and Set Working Directory

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Set directory to the uploaded component folder in Drive
project_dir = '/content/drive/MyDrive/J26-IT-349/Nidu-c1-affective'
if os.path.exists(project_dir):
    os.chdir(project_dir)
    print("Working directory set to:", os.getcwd())
    print("Contents:", os.listdir('.'))
else:
    print(f"Directory {project_dir} not found! Check your Drive path.")

## 3. Install Required Dependencies (Excluding PyTorch)

In [ ]:
# Install transformers and NLP tooling while preserving Colab's GPU PyTorch
!pip install --quiet "transformers>=4.40" "accelerate>=0.30" "datasets>=2.19" "scikit-learn>=1.4" "vaderSentiment>=3.3.2" pyarrow seaborn pyyaml

## 4. Run Full DistilRoBERTa Training (All Proxy Data)
This trains on 47,284 samples across 3 epochs, tunes per-class thresholds on `val.csv`, and evaluates `test_proxy.csv`.

In [ ]:
!python -m src.train --epochs 3 --batch-size 32 --lr 2e-5 --seed 42

## 5. Run Ablation Study: Train on GoEmotions Only
Trains on `goemotions_proxy` only (41,813 samples), evaluating on `test_proxy.csv` to measure the impact of Emotion-4 tweets on overall and per-source performance.

In [ ]:
!python -m src.train --epochs 3 --batch-size 32 --lr 2e-5 --seed 42 --ablation goemotions_proxy

## 6. View Generated Metrics and Summary

In [ ]:
import json
with open('reports/metrics/distilroberta_full.json', 'r') as f:
    metrics = json.load(f)

print("Macro F1 (Test Proxy):", metrics['overall_test_proxy_metrics']['macro_metrics']['macro_f1'])
print("Per-Class Metrics:")
for cls, m in metrics['overall_test_proxy_metrics']['per_class_metrics'].items():
    print(f"  {cls:24s} | F1: {m['f1']:.4f} | AUC: {m['roc_auc']:.4f}")

print("\nMetrics by Source:")
for src, sm in metrics['metrics_by_source'].items():
    print(f"  Source: {src} -> Macro F1: {sm['macro_metrics']['macro_f1']:.4f}")

## 7. Package Outputs for Download to Local Repo

In [ ]:
# Create a convenient zip bundle of the outputs to download
!zip -r colab_outputs.zip models/checkpoint-best models/thresholds.json reports/metrics reports/figures
print("Outputs archived to colab_outputs.zip. Download it from the Colab file browser and extract into components/Nidu-c1-affective/")